In [ ]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, URL, Text

# Folder containing all six CSV files
DATA_FOLDER = Path(r"D:\way to stability\SQL Projects\E-commerce Sales & Customer Analytics")

# MySQL connection settings
connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="jitendra@123",
    host="localhost",
    port=3306,
    database="ecommerce_analytics"
)

engine = create_engine(connection_url)

# Find every CSV in the folder
csv_files = list(DATA_FOLDER.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {DATA_FOLDER}"
    )

for file in csv_files:
    # Example: olist_customers_dataset.csv -> customers
    table_name = file.stem

    if table_name.startswith("olist_"):
        table_name = table_name[len("olist_"):]

    if table_name.endswith("_dataset"):
        table_name = table_name[:-len("_dataset")]

    print(f"\nImporting {file.name} into {table_name}...")

    # Initially load everything as text to preserve raw values.
    df = pd.read_csv(
        file,
        dtype=str,
        keep_default_na=False
    )

    # Create or replace the table and import its rows.
    column_types = {
        column: Text()
        for column in df.columns
    }

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000,
        dtype=column_types
    )

    print(f"Imported {len(df)} rows successfully.")

print("\nAll CSV files imported!")
engine.dispose()